# Readiness tests
**Must work:** similar tickets, time estimate (`AT THE WINDOW` for inquiries, median of similar incidents otherwise), priority.
**Nice to have:** category, record source.
No cell prints ticket text. The model reads only creation-time text, never `resolved_at` or `transferred_to`.

In [ ]:
import sys
sys.path.append("../src")

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import config
from load import load_tickets

df = load_tickets(verbose=True)
RESULTS = {}

In [ ]:
# Pass thresholds (starting guesses; adjust)
MIN_ROWS = 500

TYPE_MIN_INCIDENT_RECALL = 0.85 # stage 1: incidents that are NOT shown AT THE WINDOW
TYPE_MIN_INFORMED = 0.30        # stage 1: inquiry recall + incident recall - 1 (0 = no better than always guessing one answer)

SIMILAR_MIN_LIFT = 0.10         # same-category hit rate above random retrieval

TIME_MIN_GAIN_2X = 0.05         # within-2x rate above the best constant guess
TIME_MIN_GAIN_BUCKET = 0.10     # within-1-bucket rate above the best constant guess
BUCKET_EDGES = [8, 24, 72, 168]  # hours
BUCKET_NAMES = ["<8h", "8-24h", "1-3d", "3-7d", "7d+"]
NEIGHBOR_COUNTS = (3, 5, 10, 20, 30)  # k values compared (plus config.SIMILAR_TICKETS_K)
CUTOFFS = (0.3, 0.4, 0.5, 0.6, 0.7)   # similarity cutoffs compared (plus config.SIMILARITY_CUTOFF)
TIME_MIN_COVERAGE = 0.50        # used to recommend a cutoff: share of incidents with enough close matches
BEST_MIN_HOLD = 0.75            # best-case time: share of tickets that took at least that long
SUGGEST_MIN_COVERAGE = 0.50     # share of tickets that get a suggested description
SUGGEST_MIN_CONSISTENT = 0.80   # near-identical tickets that get the same suggestion
SUGGEST_MIN_MEANING = 0.90      # suggestions that leave category, record source and ticket type unchanged
SHOW_SUGGESTIONS = 0            # >0: print this many original -> suggested pairs (shows ticket text)

COST_MISSED_INCIDENT = 3        # incident shown AT THE WINDOW
COST_EXTRA_TIME = 0.25          # inquiry given a time (acceptable)
COST_BAD_BUCKET = 1             # incident given a wrong-bucket time
E2E_MIN_COST_GAIN = 0.10        # cost must be this much below the cheaper lazy strategy

PRIORITY_DEFAULT_SHARE = 0.95   # one value this common = use a fixed default
PRIORITY_MIN_GAIN = 0.03        # top-1 above majority guess
PRIORITY_MIN_RARE_RECALL = 0.50  # recall on non-common priorities

SOFT_MIN_TOP3 = 0.80            # category / record source: right value in top 3
SOFT_MAX_BELOW_MAJORITY = 0.02  # top-1 may trail the majority guess by this much

ENTRY_CLASS_WEIGHT = None       # None = plain accuracy; "balanced" favors rare values

## 0. Setup

In [ ]:
TEXT = config.COMBINED_TEXT_COLUMN
START, CLOSED = config.TIME_COLUMN, config.CLOSED_COLUMN
assert CLOSED in df.columns, f"'{CLOSED}' not found; time estimation needs it."

# Inquiry = no INC number (CTC only), or resolved at the window (<= INQUIRY_MAX_HOURS); otherwise incident
df = df.copy()
df["hours"] = (df[CLOSED] - df[START]).dt.total_seconds() / 3600
df["valid_time"] = df["hours"].notna() & (df["hours"] >= 0)
df["has_inc"] = df[config.TICKET_ID_COLUMN].str.startswith("INC")
quick = df["valid_time"] & (df["hours"] <= config.INQUIRY_MAX_HOURS)
df["is_inquiry"] = quick | ~df["has_inc"]
df["is_incident"] = df["valid_time"] & ~df["is_inquiry"]
df["typed"] = df["is_inquiry"] | df["is_incident"]  # tickets we can label for stage 1

# Time-based split: oldest trains, newest tests
df_sorted = df.sort_values(START).reset_index(drop=True)
n_test = int(len(df_sorted) * config.TEST_FRACTION)
assert n_test >= 20, f"Test set has only {n_test} tickets."
train = df_sorted.iloc[:-n_test].reset_index(drop=True)
test = df_sorted.iloc[-n_test:].reset_index(drop=True)

print(f"Rows {len(df)} (min {MIN_ROWS}: {'OK' if len(df) >= MIN_ROWS else 'LOW'}) | train/test {len(train)}/{len(test)}")
print(f"Dates {df_sorted[START].min().date()} to {df_sorted[START].max().date()} | test starts {test[START].min().date()}")
print(f"Duplicate IDs {int(df[config.TICKET_ID_COLUMN].duplicated().sum())} | "
      f"no ID {int((df[config.TICKET_ID_COLUMN] == '(no id)').sum())}")
print(f"Text length median {int(df[TEXT].str.len().median())} chars, {(df[TEXT].str.len() < 15).mean():.1%} under 15")
print(f"Ignored for timing: {int((df['hours'] < 0).sum())} resolved before opened, {int(df['hours'].isna().sum())} no resolved time")

In [ ]:
# Column profile (counts only)
for col in [*config.LABEL_COLUMNS.values(), *[c for c in config.PROFILE_COLUMNS if c in df.columns]]:
    print(f"{col}: {df[col].isna().mean():.0%} blank, {df[col].nunique()} values")
    print(df[col].value_counts(normalize=True).head(6).round(3).to_string(), "\n")

## 1. Stage 1: General Inquiry or Incident?

In [ ]:
# 1a. Type split and what lines up with it
v = df_sorted[df_sorted["typed"]]
print(f"Inquiries {int(v['is_inquiry'].sum())} ({v['is_inquiry'].mean():.0%}) | incidents {int(v['is_incident'].sum())} ({v['is_incident'].mean():.0%})\n")
print("Inquiry share by month:")
print(v["is_inquiry"].groupby(v[START].dt.to_period("M")).mean().round(2).to_string())


def type_table(groups, top=8):
    """Share of inquiry/incident per value, with counts."""
    ct = pd.crosstab(groups, v["is_inquiry"].map({True: "inquiry", False: "incident"}))
    share = ct.div(ct.sum(axis=1), axis=0).round(2)
    share["n"] = ct.sum(axis=1)
    return share.sort_values("n", ascending=False).head(top)


has_inc = v[config.TICKET_ID_COLUMN].str.startswith("INC").map({True: "has INC number", False: "CTC only"})
print("\nType by ID type:")
print(type_table(has_inc).to_string())
if "call_type" in v.columns:
    print("\nType by call_type:")
    print(type_table(v["call_type"].fillna("(blank)")).to_string())

In [ ]:
# 1a2. Duration bands by ID type (counts only): missing times, and where quick tickets cluster
edges = sorted({round(h, 4) for h in (config.INQUIRY_MAX_HOURS, 0.05, 0.25, 1, 8)})  # hours; cutoff always an edge


def fmt(h):
    return f"{h * 60:.0f} min" if h < 1 else f"{h:.0f} h"


names = ["negative", f"<= {fmt(edges[0])}", *[f"{fmt(lo)}-{fmt(hi)}" for lo, hi in zip(edges, edges[1:])], f"{fmt(edges[-1])}+"]
band = pd.Series("no resolved time", index=df_sorted.index)
ok = df_sorted["hours"].notna()
band[ok] = pd.cut(df_sorted.loc[ok, "hours"], [-np.inf, -1e-9, *edges, np.inf], labels=names).astype(str)
id_type = df_sorted[config.TICKET_ID_COLUMN].str.startswith("INC").map({True: "has INC number", False: "CTC only"})
print(pd.crosstab(id_type, band).reindex(columns=["no resolved time", *names], fill_value=0).to_string())

In [ ]:
# 1b. Can text tell them apart?
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tr_v = train[train["typed"]].reset_index(drop=True)
te_v = test[test["typed"]].reset_index(drop=True)
ytr1 = tr_v["is_inquiry"].to_numpy()
yte1 = te_v["is_inquiry"].to_numpy()
assert len(np.unique(ytr1)) == 2, "Training data has only one ticket type."

vec_1 = TfidfVectorizer(ngram_range=config.TFIDF_WORD_NGRAMS, min_df=config.TFIDF_MIN_DF,
                        max_features=config.TFIDF_MAX_FEATURES, sublinear_tf=True)
Xtr1 = vec_1.fit_transform(tr_v[TEXT])
Xte1 = vec_1.transform(te_v[TEXT])
clf1 = LogisticRegression(C=config.REGULARIZATION_C, class_weight=config.CLASS_WEIGHT,
                          max_iter=config.MAX_ITER).fit(Xtr1, ytr1)
p1 = clf1.predict_proba(Xte1)[:, list(clf1.classes_).index(True)]
te_v["pred_inquiry"] = p1 >= config.INQUIRY_CUTOFF
pred1 = te_v["pred_inquiry"].to_numpy()

majority_is_inquiry = ytr1.mean() >= 0.5
acc = float((pred1 == yte1).mean())
acc_base = float((yte1 == majority_is_inquiry).mean())
recall_inq = float(pred1[yte1].mean()) if yte1.any() else float("nan")
recall_inc = float((~pred1[~yte1]).mean()) if (~yte1).any() else float("nan")
balanced = (recall_inq + recall_inc) / 2

conf = pd.DataFrame(
    [[int((pred1 & yte1).sum()), int((~pred1 & yte1).sum())],
     [int((pred1 & ~yte1).sum()), int((~pred1 & ~yte1).sum())]],
    index=["actual inquiry", "actual incident"], columns=["predicted inquiry", "predicted incident"])
print(f"Test {len(yte1)} (inquiries {int(yte1.sum())}, incidents {int((~yte1).sum())})\n")
print(conf.to_string(), "\n")
print(f"Accuracy {acc:.0%} (always '{'inquiry' if majority_is_inquiry else 'incident'}': {acc_base:.0%})")
print(f"Inquiries caught {recall_inq:.0%} | incidents caught {recall_inc:.0%} | balanced {balanced:.0%}")

# Genuine skill vs a constant guess: always saying "incident" catches 100% of incidents and 0% of inquiries.
from sklearn.metrics import roc_auc_score
informed = recall_inq + recall_inc - 1
auc = float(roc_auc_score(yte1, p1))
print(f"Always guessing 'incident': incidents caught 100%, inquiries caught 0%, informedness 0%")
print(f"This model: informedness {informed:.0%} (0% = no better than a constant guess), AUC {auc:.2f} (0.50 = coin flip, independent of the cutoff)")

RESULTS["stage 1: inquiry vs incident"] = {
    "tier": "MUST",
    "pass": recall_inc >= TYPE_MIN_INCIDENT_RECALL and informed >= TYPE_MIN_INFORMED,
    "detail": f"incidents caught {recall_inc:.0%} (need {TYPE_MIN_INCIDENT_RECALL:.0%}), informedness {informed:.0%} (need {TYPE_MIN_INFORMED:.0%}; 0% = constant guess), AUC {auc:.2f}",
}

In [ ]:
# 1c. Where stage 1 misses: ID type, and how the mix shifts over time (counts only)
te_v["ctc_only"] = ~te_v[config.TICKET_ID_COLUMN].str.startswith("INC")
rows = []
for name, m in (("INC tickets", ~te_v["ctc_only"]), ("CTC only", te_v["ctc_only"])):
    y, p = yte1[m], pred1[m]
    rows.append({"group": name, "tickets": int(m.sum()), "inquiries": int(y.sum()), "incidents": int((~y).sum()),
                 "inquiries caught": f"{p[y].mean():.0%}" if y.any() else "-",
                 "incidents caught": f"{(~p[~y]).mean():.0%}" if (~y).any() else "-"})
print(pd.DataFrame(rows).to_string(index=False), "\n")

ty = df[df["typed"]].copy()
ty["month"] = ty[config.TIME_COLUMN].dt.to_period("M")
ty["ctc_only"] = ~ty[config.TICKET_ID_COLUMN].str.startswith("INC")
mix = ty.groupby("month").agg(tickets=("typed", "size"), ctc_only=("ctc_only", "sum"), inquiries=("is_inquiry", "sum"))
mix["incidents"] = mix["tickets"] - mix["inquiries"]
mix["ctc_only %"] = (100 * mix["ctc_only"] / mix["tickets"]).round()
mix["incident %"] = (100 * mix["incidents"] / mix["tickets"]).round()
print(mix.to_string())

In [ ]:
# 1d. Cutoff sweep: call "inquiry" only when sure. Higher = fewer incidents shown AT THE WINDOW.
p_inq = clf1.predict_proba(Xte1)[:, list(clf1.classes_).index(True)]
rows = []
for t in (0.5, 0.6, 0.7, 0.8, 0.9):
    p = p_inq >= t
    ri, rn = p[yte1].mean(), (~p[~yte1]).mean()
    rows.append({"cutoff": t, "inquiries caught": f"{ri:.0%}", "incidents caught": f"{rn:.0%}", "informedness": f"{ri + rn - 1:.0%}",
                 "incidents shown AT THE WINDOW": int((p & ~yte1).sum()), "inquiries given a time": int((~p & yte1).sum())})
print(pd.DataFrame(rows).to_string(index=False), "\n")

# CTC-only tickets predicted as incidents (IDs only): open these in ServiceNow
MAX_IDS = 40
miss = te_v[te_v["ctc_only"] & ~te_v["pred_inquiry"]].assign(p_inquiry=p_inq[(te_v["ctc_only"] & ~te_v["pred_inquiry"]).to_numpy()])
miss = miss.sort_values("p_inquiry")
print(f"CTC-only predicted incident: {len(miss)} (lowest inquiry probability first, showing {min(MAX_IDS, len(miss))})")
for _, r in miss.head(MAX_IDS).iterrows():
    print(f"  {r[config.TICKET_ID_COLUMN]}  p(inquiry)={r['p_inquiry']:.2f}")

## 2. Similar tickets and incident time estimate
Estimate = median time of the most similar past incidents. Similar-ticket quality is scored outside the dominant category.

In [ ]:
def top_k_similar(Xq, Xdb, k, chunk=500):
    """Top-k indices and scores of Xdb rows per Xq row (sparse or dense, L2-normalized)."""
    idx_out, sc_out = [], []
    for start in range(0, Xq.shape[0], chunk):
        S = Xq[start:start + chunk] @ Xdb.T
        S = S.toarray() if hasattr(S, "toarray") else np.asarray(S)
        kk = min(k, S.shape[1])
        part = np.argpartition(-S, kk - 1, axis=1)[:, :kk]
        part_sc = np.take_along_axis(S, part, axis=1)
        order = np.argsort(-part_sc, axis=1)
        idx_out.append(np.take_along_axis(part, order, axis=1))
        sc_out.append(np.take_along_axis(part_sc, order, axis=1))
    return np.vstack(idx_out), np.vstack(sc_out)


def within_factor(pred, actual, factor=2.0, floor=0.25):
    """Share of predictions within `factor`x of actual (times floored at 15 min)."""
    ratio = np.maximum(pred, floor) / np.maximum(actual, floor)
    return float(((ratio >= 1 / factor) & (ratio <= factor)).mean())


def typical_miss(pred, actual, floor=0.25):
    """Median multiplicative miss (1 = exact, 2 = off by 2x)."""
    p, a = np.maximum(pred, floor), np.maximum(actual, floor)
    return float(np.median(np.maximum(p / a, a / p)))


def pooled(hist, sc, cut, k, qb):
    """Per ticket: median, 25th and 75th percentile and best case of matches >= cut (up to SIMILAR_MAX).
    Fewer than SIMILAR_MIN close matches: use the nearest k and flag it."""
    keep = sc >= cut
    keep[:, config.SIMILAR_MAX:] = False
    few = keep.sum(axis=1) < config.SIMILAR_MIN
    keep[few] = False
    keep[few, :k] = True
    nb = np.where(keep, hist, np.nan)
    return (np.nanmedian(nb, axis=1), np.nanpercentile(nb, 25, axis=1), np.nanpercentile(nb, 75, axis=1),
            np.nanquantile(nb, qb, axis=1), few, keep.sum(axis=1))


def similar_and_time_tests(name, Xtr_all, Xte_all):
    """Similar-ticket and time-estimate tests for one text representation."""
    # (a) similar-ticket quality outside the dominant category (labeled tickets only)
    lab = config.TARGET_COLUMN
    tr_has = train[lab].notna().to_numpy()
    te_has = test[lab].notna().to_numpy()
    tr_lab = train.loc[tr_has, lab].astype(str).to_numpy()
    te_lab = test[lab].fillna("").astype(str).to_numpy()
    shares = pd.Series(tr_lab).value_counts(normalize=True)
    sel = np.where(te_has & (te_lab != shares.index[0]))[0]
    print(f"[{name}] Similar tickets: {len(sel)} labeled test tickets outside dominant '{lab}' ({shares.iloc[0]:.0%} of labeled training)")
    if len(sel) >= 20:
        idx3, _ = top_k_similar(Xte_all[sel], Xtr_all[tr_has], config.SIMILAR_SHOWN)
        match = tr_lab[idx3] == te_lab[sel][:, None]
        hit1, hit3 = float(match[:, 0].mean()), float(match.any(axis=1).mean())
        p = shares.reindex(te_lab[sel]).fillna(0).to_numpy()
        chance = float((1 - (1 - p) ** config.SIMILAR_SHOWN).mean())
        print(f"    same-'{lab}' hit: top-1 {hit1:.0%}, top-{config.SIMILAR_SHOWN} {hit3:.0%} (random {chance:.0%})")
        RESULTS[f"similar tickets ({name})"] = {
            "tier": "MUST",
            "pass": hit3 >= chance + SIMILAR_MIN_LIFT,
            "detail": f"top-{config.SIMILAR_SHOWN} hit {hit3:.0%} vs random {chance:.0%} (need +{SIMILAR_MIN_LIFT:.0%}), non-dominant tickets",
        }
    else:
        print("    too few to score; use the spot check.")

    # (b) median-of-similar-incidents time estimate
    tr_inc = train["is_incident"].to_numpy()
    te_inc = test["is_incident"].to_numpy()
    db, q = Xtr_all[tr_inc], Xte_all[te_inc]
    hrs_db = train.loc[tr_inc, "hours"].to_numpy()
    actual = test.loc[te_inc, "hours"].to_numpy()
    assert len(actual) >= 20, f"Only {len(actual)} incidents in the test set."

    ks = sorted(set(NEIGHBOR_COUNTS) | {config.SIMILAR_TICKETS_K})
    kmax = min(max(*ks, config.SIMILAR_MAX), db.shape[0])
    idxk, sck = top_k_similar(q, db, kmax)
    b_true = np.digitize(actual, BUCKET_EDGES)

    def metrics(label, est, lo=None, hi=None):
        b_est = np.digitize(est, BUCKET_EDGES)
        return {
            "estimate": label,
            "median error (h)": round(float(np.median(np.abs(est - actual))), 1),
            "typical miss (x)": round(typical_miss(est, actual), 1),
            "within 2x": within_factor(est, actual),
            "exact bucket": float((b_est == b_true).mean()),
            "within 1 bucket": float((np.abs(b_est - b_true) <= 1).mean()),
            "range holds actual": float(((actual >= lo) & (actual <= hi)).mean()) if lo is not None else np.nan,
        }

    # Constant-guess baselines: typical time, best single time, most common bucket
    cands = np.quantile(hrs_db, np.linspace(0.05, 0.95, 19))
    best_c = max(cands, key=lambda c: within_factor(np.full(len(hrs_db), c), hrs_db))
    bk = np.digitize(hrs_db, BUCKET_EDGES)
    b_major = int(np.bincount(bk, minlength=len(BUCKET_NAMES)).argmax())
    baselines = [
        (f"baseline: always {np.median(hrs_db):.1f} h (median)", np.median(hrs_db)),
        (f"baseline: always {best_c:.1f} h (best single time)", best_c),
        (f"baseline: always '{BUCKET_NAMES[b_major]}' bucket", np.median(hrs_db[bk == b_major])),
    ]
    rows = [metrics(label, np.full(len(actual), val)) for label, val in baselines]
    chosen = None
    for k in ks:
        if k > kmax:
            continue
        nb = hrs_db[idxk[:, :k]]
        row = metrics(f"median of {k} similar", np.median(nb, axis=1),
                      np.percentile(nb, 25, axis=1), np.percentile(nb, 75, axis=1))
        rows.append(row)

    table = pd.DataFrame(rows)
    for c in ("within 2x", "exact bucket", "within 1 bucket", "range holds actual"):
        table[c] = table[c].map(lambda x: "" if pd.isna(x) else f"{x:.0%}")
    print(f"\n[{name}] Fixed-count estimate, {len(actual)} test incidents (range = middle half of similar times):")
    print(table.to_string(index=False))

    # Similarity cutoff: every past incident at least `cut` similar (up to SIMILAR_MAX).
    # Fewer than SIMILAR_MIN close matches: nearest-k fallback. Best case = BEST_CASE_QUANTILE of the matches.
    def score(est, a):
        be, ba = np.digitize(est, BUCKET_EDGES), np.digitize(a, BUCKET_EDGES)
        return within_factor(est, a), float((np.abs(be - ba) <= 1).mean()), float((a >= est).mean())

    kf = min(config.SIMILAR_TICKETS_K, kmax)
    hist = hrs_db[idxk]
    qb = config.BEST_CASE_QUANTILE
    crows, chosen, ok_cuts = [], None, []
    for cut in sorted(set(CUTOFFS) | {config.SIMILARITY_CUTOFF}):
        med, _, _, best, few, used = pooled(hist, sck, cut, kf, qb)
        cov = ~few
        if cov.mean() >= TIME_MIN_COVERAGE:
            ok_cuts.append(cut)
        a2, a1, _ = score(med, actual)
        _, _, ah = score(best, actual)
        row = {"cutoff": f"{cut:.2f}", "close matches": f"{cov.mean():.0%}", "avg used": f"{used[cov].mean():.1f}" if cov.any() else ""}
        if cov.sum() >= 20:
            m2, m1, _ = score(med[cov], actual[cov])
            base = [score(np.full(int(cov.sum()), v), actual[cov]) for _, v in baselines]
            row.update({"close: 2x": f"{m2:.0%}", "close: 1 bucket": f"{m1:.0%}",
                        "close baseline: 2x": f"{max(x[0] for x in base):.0%}", "close baseline: 1 bucket": f"{max(x[1] for x in base):.0%}"})
        row.update({"all: 2x": f"{a2:.0%}", "all: 1 bucket": f"{a1:.0%}", "all: best case took at least": f"{ah:.0%}"})
        crows.append(row)
        if cut == config.SIMILARITY_CUTOFF:
            chosen = {"cov": float(cov.mean()), "a2": a2, "a1": a1, "ah": ah}
    base_all = {"2x": max(r["within 2x"] for r in rows[:3]), "1b": max(r["within 1 bucket"] for r in rows[:3])}
    print(f"\n[{name}] Similarity cutoff. close = tickets with enough close matches; all = every incident, nearest {kf} where there are too few.")
    print(f"  Best case = {qb:.0%} of matches finished faster; 'took at least' = share of tickets that took that long or more.")
    print(f"  Strongest constant guess on all incidents: within 2x {base_all['2x']:.0%}, within 1 bucket {base_all['1b']:.0%}")
    print(pd.DataFrame(crows).fillna("").to_string(index=False))
    print(f"  Highest cutoff with >= {TIME_MIN_COVERAGE:.0%} close matches: {max(ok_cuts):.2f}" if ok_cuts
          else f"  No cutoff here reaches {TIME_MIN_COVERAGE:.0%} close matches.")

    if chosen:
        g2, g1 = chosen["a2"] - base_all["2x"], chosen["a1"] - base_all["1b"]
        RESULTS[f"time estimate: incidents ({name})"] = {
            "tier": "MUST",
            "pass": g2 >= TIME_MIN_GAIN_2X and g1 >= TIME_MIN_GAIN_BUCKET,
            "detail": f"all >= {config.SIMILARITY_CUTOFF:.2f} similar ({chosen['cov']:.0%} close, rest nearest {kf}): within 2x {chosen['a2']:.0%} vs {base_all['2x']:.0%} best baseline "
                      f"(need +{TIME_MIN_GAIN_2X:.0%}), within-1-bucket {chosen['a1']:.0%} vs {base_all['1b']:.0%} (need +{TIME_MIN_GAIN_BUCKET:.0%})",
        }
        RESULTS[f"time estimate: best case ({name})"] = {
            "tier": "NICE",
            "pass": chosen["ah"] >= BEST_MIN_HOLD,
            "detail": f"{qb:.0%} best case: ticket took at least that long {chosen['ah']:.0%} (need {BEST_MIN_HOLD:.0%})",
        }

In [ ]:
# 2a. TF-IDF
vec_s = TfidfVectorizer(ngram_range=config.TFIDF_WORD_NGRAMS, min_df=config.TFIDF_MIN_DF,
                        max_features=config.TFIDF_MAX_FEATURES, sublinear_tf=True)
Xtr_s = vec_s.fit_transform(train[TEXT])
Xte_s = vec_s.transform(test[TEXT])
similar_and_time_tests("tf-idf", Xtr_s, Xte_s)

In [ ]:
# 2b. Spot check (IDs and numbers only): look a few up in ServiceNow
def same_label(a, b):
    """True when both labels are present and equal."""
    return bool(pd.notna(a) and pd.notna(b) and a == b)


def kind(row):
    """'inquiry', 'incident, N h', or 'no resolved time'."""
    if row["is_inquiry"]:
        return "inquiry"
    return "incident, %.1f h" % row["hours"] if row["valid_time"] else "no resolved time"


rng = np.random.default_rng(config.RANDOM_SEED)
idx_s, sc_s = top_k_similar(Xte_s, Xtr_s, config.SIMILAR_SHOWN)
for i in rng.choice(len(test), size=5, replace=False):
    t = test.loc[i]
    print(f"Ticket {t[config.TICKET_ID_COLUMN]} [{t[config.TARGET_COLUMN]}] ({kind(t)})")
    for j, s in zip(idx_s[i], sc_s[i]):
        m = train.loc[j]
        same = "same category" if same_label(m[config.TARGET_COLUMN], t[config.TARGET_COLUMN]) else "different/blank category"
        print(f"    -> {m[config.TICKET_ID_COLUMN]}  sim {s:.2f}  {same}  ({kind(m)})")

In [ ]:
# 2c. Optional: MiniLM comparison (skipped if not downloaded)
mdir = config.model_dir("minilm")
if not mdir.exists():
    print(f"Skipped: {mdir} not found. Run src/download_models.py.")
else:
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer(str(mdir))
    Etr = enc.encode(train[TEXT].tolist(), batch_size=config.EMBED_BATCH_SIZE,
                     normalize_embeddings=True, show_progress_bar=False)
    Ete = enc.encode(test[TEXT].tolist(), batch_size=config.EMBED_BATCH_SIZE,
                     normalize_embeddings=True, show_progress_bar=False)
    similar_and_time_tests("minilm", Etr, Ete)

## 3. End to end: what the technician sees
Mistakes by cost: an incident shown `AT THE WINDOW` (costly), an incident given a wrong-bucket time, and an inquiry given a time (cheap).
Scored as weighted cost against two lazy strategies: always `AT THE WINDOW`, and always show a time.

In [ ]:
k = config.SIMILAR_TICKETS_K
pred_inq = te_v["pred_inquiry"].to_numpy()
true_inq = te_v["is_inquiry"].to_numpy()
true_inc = ~true_inq
routed = ~pred_inq  # gets a time estimate
n_inc, n_inq = int(true_inc.sum()), int(true_inq.sum())

Xte_v = vec_s.transform(te_v[TEXT])
tr_inc_mask = train["is_incident"].to_numpy()
db_inc = Xtr_s[tr_inc_mask]
hrs_inc = train.loc[tr_inc_mask, "hours"].to_numpy()

kk = min(max(k, config.SIMILAR_MAX), db_inc.shape[0])
idx_a, sc_a = top_k_similar(Xte_v, db_inc, kk)
est, lo, hi, _, fallback, _ = pooled(hrs_inc[idx_a], sc_a, config.SIMILARITY_CUTOFF, min(k, kk), config.BEST_CASE_QUANTILE)

hours_true = te_v["hours"].to_numpy()
bucket_ok = np.zeros(len(te_v), dtype=bool)  # incident estimate in the right or an adjacent bucket
bucket_ok[true_inc] = np.abs(np.digitize(est[true_inc], BUCKET_EDGES) - np.digitize(hours_true[true_inc], BUCKET_EDGES)) <= 1

missed = pred_inq & true_inc
extra = routed & true_inq
bad_time = routed & true_inc & ~bucket_ok
cost = COST_MISSED_INCIDENT * missed.sum() + COST_EXTRA_TIME * extra.sum() + COST_BAD_BUCKET * bad_time.sum()
cost_window = COST_MISSED_INCIDENT * n_inc                                       # always AT THE WINDOW
cost_time = COST_EXTRA_TIME * n_inq + COST_BAD_BUCKET * (true_inc & ~bucket_ok).sum()  # always show a time
lazy = min(cost_window, cost_time)
right = (pred_inq & true_inq) | (routed & true_inc & bucket_ok)

print(f"Test {len(te_v)} (inquiries {n_inq}, incidents {n_inc}) | right overall {right.mean():.0%}")
print(f"Incidents wrongly shown AT THE WINDOW: {int(missed.sum())} of {n_inc}  <- costly (x{COST_MISSED_INCIDENT})")
print(f"Inquiries given a time: {int(extra.sum())} of {n_inq}  (x{COST_EXTRA_TIME}, acceptable)")
print(f"Incidents given a wrong-bucket time: {int(bad_time.sum())} of {n_inc}")
print(f"Weighted cost: model {cost:.0f} | always AT THE WINDOW {cost_window:.0f} | always a time {cost_time:.0f}")
if routed.any():
    print(f"Routed tickets using the nearest {k} (too few close matches): {int(fallback[routed].sum())} of {int(routed.sum())}")

RESULTS["end to end: what the technician sees"] = {
    "tier": "MUST",
    "pass": cost <= (1 - E2E_MIN_COST_GAIN) * lazy,
    "detail": f"weighted cost {cost:.0f} vs {lazy:.0f} for the cheaper lazy strategy (need {E2E_MIN_COST_GAIN:.0%} lower); "
              f"{int(missed.sum())} incidents shown AT THE WINDOW, {int(extra.sum())} inquiries given a time",
}

rows = []
for i in rng.choice(len(te_v), size=min(10, len(te_v)), replace=False):
    shown = config.AT_THE_WINDOW_LABEL if pred_inq[i] else f"~{est[i]:.0f} h ({lo[i]:.0f}-{hi[i]:.0f} h)"
    rows.append({"ticket": te_v.loc[i, config.TICKET_ID_COLUMN], "tool shows": shown,
                 "actual": "at the window" if true_inq[i] else f"{hours_true[i]:.0f} h",
                 "right?": "yes" if right[i] else "NO"})
print()
print(pd.DataFrame(rows).to_string(index=False))

## 4. Priority (strict)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support


def predict_field(col):
    """Train on older tickets, score on newer ones; returns metrics or None."""
    tr_ok = train[col].notna().to_numpy()
    te_ok = test[col].notna().to_numpy()
    ytr = train.loc[tr_ok, col].astype(str)
    yte = test.loc[te_ok, col].astype(str).to_numpy()
    if ytr.nunique() < 2 or len(yte) == 0:
        return None
    clf = LogisticRegression(C=config.REGULARIZATION_C, class_weight=ENTRY_CLASS_WEIGHT,
                             max_iter=config.MAX_ITER).fit(Xtr_s[tr_ok], ytr)
    proba = clf.predict_proba(Xte_s[te_ok])
    ranked = clf.classes_[np.argsort(-proba, axis=1)[:, :3]]
    majority = ytr.value_counts().index[0]
    rare = yte != majority
    return {
        "yte": yte,
        "ranked": ranked,
        "majority": majority,
        "majority_share": float((ytr == majority).mean()),
        "top1": float((ranked[:, 0] == yte).mean()),
        "top3": float((ranked == yte[:, None]).any(axis=1).mean()),
        "maj_acc": float((yte == majority).mean()),
        "rare_n": int(rare.sum()),
        "rare_recall": float((ranked[rare, 0] == yte[rare]).mean()) if rare.any() else float("nan"),
        "unseen": float((~pd.Series(yte).isin(clf.classes_)).mean()),
    }


for name in config.ENTRY_FIELDS_STRICT:
    col = config.LABEL_COLUMNS[name]
    r = predict_field(col)
    if r is None:
        print(f"Skipped '{col}': not enough variety or no test labels.")
        continue
    print(f"== {col} == ({len(r['yte'])} test tickets)")
    print(f"Common value '{r['majority']}' = {r['majority_share']:.0%} of training")
    print(f"Top-1 {r['top1']:.0%} (default {r['maj_acc']:.0%}) | top-3 {r['top3']:.0%}")
    print(f"Uncommon values: {r['rare_n']} test tickets, {r['rare_recall']:.0%} right")

    labels = sorted(set(r["yte"]))
    p, rc, _, n = precision_recall_fscore_support(r["yte"], r["ranked"][:, 0], labels=labels, zero_division=0)
    print(pd.DataFrame({"value": labels, "test tickets": n, "recall": rc.round(2), "precision": p.round(2)}).to_string(index=False))

    rare_ok = r["rare_n"] == 0 or r["rare_recall"] >= PRIORITY_MIN_RARE_RECALL
    if r["majority_share"] >= PRIORITY_DEFAULT_SHARE:
        ok = rare_ok
        detail = (f"'{r['majority']}' is {r['majority_share']:.0%}: use a fixed default. "
                  f"Uncommon-value recall {r['rare_recall']:.0%} (need {PRIORITY_MIN_RARE_RECALL:.0%} for a model to help)")
    else:
        ok = (r["top1"] - r["maj_acc"]) >= PRIORITY_MIN_GAIN and rare_ok
        detail = (f"top-1 {r['top1']:.0%} vs {r['maj_acc']:.0%} default (need +{PRIORITY_MIN_GAIN:.0%}), "
                  f"uncommon-value recall {r['rare_recall']:.0%} (need {PRIORITY_MIN_RARE_RECALL:.0%})")
    RESULTS[f"entry: {col}"] = {"tier": "MUST", "pass": ok, "detail": detail}

## 5. Category and record source (loose)

In [ ]:
rows = []
for name in config.ENTRY_FIELDS_SOFT:
    col = config.LABEL_COLUMNS[name]
    r = predict_field(col)
    if r is None:
        print(f"Skipped '{col}': not enough variety or no test labels.")
        continue
    ok = r["top3"] >= SOFT_MIN_TOP3 and r["top1"] >= r["maj_acc"] - SOFT_MAX_BELOW_MAJORITY
    RESULTS[f"entry: {col}"] = {
        "tier": "NICE",
        "pass": ok,
        "detail": f"top-1 {r['top1']:.0%} (default {r['maj_acc']:.0%}), top-3 {r['top3']:.0%} (need {SOFT_MIN_TOP3:.0%})",
    }
    rows.append({
        "field": col,
        "top-1": f"{r['top1']:.0%}",
        "top-3": f"{r['top3']:.0%}",
        "common-value default": f"{r['maj_acc']:.0%}",
        "uncommon caught": "n/a" if np.isnan(r["rare_recall"]) else f"{r['rare_recall']:.0%}",
        "unseen in training": f"{r['unseen']:.0%}",
    })
pd.DataFrame(rows)

## 6. Suggested short description
Most typical wording among close past tickets, using only words that recur across those tickets (names and one-off details drop out).

In [ ]:
import re
from collections import Counter

PII = [re.compile(p) for p in (config.EID_PATTERN, config.EMAIL_PATTERN, config.PHONE_PATTERN)]
tr_t = train["typed"].to_numpy()
texts_tr = train.loc[tr_t, TEXT].to_numpy()
words_tr = [set(re.findall(r"[a-z0-9]+", t.lower())) for t in texts_tr]
Xtr_t = Xtr_s[tr_t]
Xq = vec_s.transform(te_v[TEXT])


def suggest(idx, sc):
    """Most typical wording among close matches, or None."""
    nb = idx[sc >= config.SUGGEST_CUTOFF]
    if len(nb) < config.SUGGEST_MIN:
        return None
    seen = Counter(w for i in nb for w in words_tr[i])
    need = max(config.SUGGEST_WORD_MIN_COUNT, int(np.ceil(config.SUGGEST_WORD_SHARE * (len(nb) - 1))))
    S = (Xtr_t[nb] @ Xtr_t[nb].T).toarray()
    typical = (S.sum(axis=1) - 1) / (len(nb) - 1)
    for j in sorted(range(len(nb)), key=lambda j: (-round(typical[j], 4), len(texts_tr[nb[j]]))):
        i = nb[j]
        if words_tr[i] and all(seen[w] - 1 >= need for w in words_tr[i]) and not any(p.search(texts_tr[i]) for p in PII):
            return texts_tr[i]
    return None


idx_s, sc_s = top_k_similar(Xq, Xtr_t, min(config.SUGGEST_MAX, Xtr_t.shape[0]))
sugg = [suggest(i, s) for i, s in zip(idx_s, sc_s)]
has = np.array([s is not None for s in sugg])
orig = te_v[TEXT].to_numpy()
cover = float(has.mean())
print(f"Test {len(te_v)} | suggestion for {int(has.sum())} ({cover:.0%})")

if has.sum() < 20:
    RESULTS["suggested description"] = {"tier": "NICE", "pass": False, "detail": f"only {int(has.sum())} tickets get a suggestion"}
else:
    # Consistency: each ticket vs its closest other test ticket
    S_tt = (Xq @ Xq.T).toarray()
    np.fill_diagonal(S_tt, 0)
    nn = S_tt.argmax(axis=1)
    pair = has & has[nn] & (S_tt.max(axis=1) >= config.SUGGEST_CUTOFF)
    pi = np.where(pair)[0]
    same_s = float(np.mean([sugg[i] == sugg[nn[i]] for i in pi])) if len(pi) else float("nan")
    same_o = float(np.mean([orig[i] == orig[nn[i]] for i in pi])) if len(pi) else float("nan")
    print(f"Near-identical pairs ({len(pi)}): same suggestion {same_s:.0%} (same original wording {same_o:.0%})")
    print(f"Distinct wordings among these tickets: {len(set(orig[has]))} originals -> {len(set(np.array(sugg, dtype=object)[has]))} suggestions")

    # Meaning preserved: model answers on the suggestion vs the original text
    sg = [s for s in sugg if s is not None]
    Xs = vec_s.transform(sg)
    same = {}
    for col in (config.LABEL_COLUMNS[f] for f in (*config.ENTRY_FIELDS_SOFT,)):
        ok = train[col].notna().to_numpy()
        ytr_f = train.loc[ok, col].astype(str)
        if ytr_f.nunique() < 2:
            continue
        clf_f = LogisticRegression(C=config.REGULARIZATION_C, class_weight=ENTRY_CLASS_WEIGHT,
                                   max_iter=config.MAX_ITER).fit(Xtr_s[ok], ytr_f)
        same[col] = clf_f.predict(Xq[has]) == clf_f.predict(Xs)
    col_true = list(clf1.classes_).index(True)
    p_orig = te_v["pred_inquiry"].to_numpy()[has]
    p_sug = clf1.predict_proba(vec_1.transform(sg))[:, col_true] >= config.INQUIRY_CUTOFF
    same["inquiry vs incident"] = p_orig == p_sug
    for name_, v in same.items():
        print(f"Unchanged {name_}: {v.mean():.0%}")
    meaning = float(np.logical_and.reduce(list(same.values())).mean())
    print(f"All unchanged: {meaning:.0%}")

    pii_hits = sum(any(p.search(s) for p in PII) for s in sg)
    print(f"Suggestions with an EID, email or phone pattern: {pii_hits}")

    RESULTS["suggested description"] = {
        "tier": "NICE",
        "pass": cover >= SUGGEST_MIN_COVERAGE and same_s >= SUGGEST_MIN_CONSISTENT and meaning >= SUGGEST_MIN_MEANING and pii_hits == 0,
        "detail": f"{cover:.0%} covered (need {SUGGEST_MIN_COVERAGE:.0%}), same suggestion for near-identical tickets {same_s:.0%} "
                  f"(need {SUGGEST_MIN_CONSISTENT:.0%}), meaning unchanged {meaning:.0%} (need {SUGGEST_MIN_MEANING:.0%}), PII hits {pii_hits}",
    }

    if SHOW_SUGGESTIONS:
        for i in rng.choice(np.where(has)[0], size=min(SHOW_SUGGESTIONS, int(has.sum())), replace=False):
            print(f"\n{te_v.loc[i, config.TICKET_ID_COLUMN]}\n  was:  {orig[i]}\n  now:  {sugg[i]}")

## Summary

In [ ]:
print(f"{'RESULT':7} {'TIER':5} {'TEST':44} DETAIL")
print("-" * 120)
for name, r in sorted(RESULTS.items(), key=lambda kv: (kv[1]["tier"] != "MUST", kv[0])):
    print(f"{'PASS' if r['pass'] else 'FAIL':7} {r['tier']:5} {name:44} {r['detail']}")